# 08 -- Improved Evaluation: Balanced Sampler + Prototype+TTA

**Changes from v7**:

| # | Change | Effect |
|---|--------|--------|
| 1 | Dataset-balanced batch sampler | Each dataset contributes equally per batch (was 87% private) |
| 2 | Margin curriculum 0.1 → **0.70**, ramp 20 ep | Tighter clusters for 3000-class setting |
| 3 | Longer training: **40 / 20 / 15** epochs | More iterations for the larger training set |
| 4 | Phase 3: **batch-hard triplet mining** | Mines hardest genuine-forgery pairs per batch instead of static pairs |
| 5 | Evaluation: **Prototype + TTA** (N=8 views) | Writer prototype = mean of enrolled genuine embeds; TTA reduces single-image noise |

**Primary target**: Private dataset EER (thesis evaluation).
**All datasets** kept for universal signature style coverage.


---
## Section 1 -- Imports & Config

In [ ]:
import sys, re, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Sampler
import torchvision.transforms as T
import torchvision.models as models

from sklearn import metrics as sk_metrics
from sklearn.decomposition import PCA

def _find_project_root(start):
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'requirements.txt').exists():
            return p
    raise RuntimeError(f'Cannot find project root from {start}')

PROJECT_ROOT = _find_project_root(Path().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.metrics.verification import compute_metrics

SEED         = 42
set_seed(SEED)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DS_ROOT      = PROJECT_ROOT / 'datasets'
EMB_DIM      = 256
K_SUB        = 3        # SubCenterArcFace sub-centres per writer
IMG_SIZE     = 224
BATCH        = 64
N_P1         = 40       # Phase 1 epochs (was 30)
N_P2         = 20       # Phase 2 epochs (was 15)
N_P3         = 15       # Phase 3 epochs (was 10)
WARMUP       = 3
MARGIN_START = 0.10
MARGIN_END   = 0.70     # was 0.50
RAMP_EPOCHS  = 20       # was 15
TTA_N        = 8        # TTA augmented views at inference
K_ENROLL     = 6        # genuine samples used to build writer prototype

# Phase 3 batch structure
N_WRITERS_P3 = 10       # writers per Phase-3 batch
N_GEN_P3     = 3        # genuine images per writer per batch
N_FORG_P3    = 2        # forgery images per writer per batch
BATCH_P3     = N_WRITERS_P3 * (N_GEN_P3 + N_FORG_P3)  # = 50

print('Device      :', DEVICE)
print('Dataset root:', DS_ROOT)

---
## Section 2 -- Dataset Scanners (same as v7)

SigComp2011 forgery fix retained: `r'^\d{4}(\d{3})_'` maps to target writer.
Private: `original_{id}_{n}.jpg` / `forgeries_{id}_{n}.jpg`.


In [ ]:
def scan_cedar(root):
    root  = Path(root)
    pat_o = re.compile(r'^original_(\d+)_(\d+)\.png$',  re.I)
    pat_f = re.compile(r'^forgeries_(\d+)_(\d+)\.png$', re.I)
    rows  = []
    for fp in (root/'full_org').iterdir():
        m = pat_o.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'cedar_{int(m.group(1)):03d}','label':'genuine','dataset':'cedar'})
    for fp in (root/'full_forg').iterdir():
        m = pat_f.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'cedar_{int(m.group(1)):03d}','label':'forgery','dataset':'cedar'})
    return pd.DataFrame(rows, columns=['path','writer_uid','label','dataset'])

def scan_gpds150(root):
    root  = Path(root)
    pat_g = re.compile(r'^c-(\d+)-',  re.I)
    pat_f = re.compile(r'^cf-(\d+)-', re.I)
    rows  = []
    for fp in (root/'train'/'genuine').iterdir():
        m = pat_g.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'gpds_{int(m.group(1)):03d}','label':'genuine','dataset':'gpds150'})
    for fp in (root/'train'/'forge').iterdir():
        m = pat_f.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'gpds_{int(m.group(1)):03d}','label':'forgery','dataset':'gpds150'})
    return pd.DataFrame(rows, columns=['path','writer_uid','label','dataset'])

def scan_sigcomp2011(root):
    root     = Path(root)
    pat_gen  = re.compile(r'^(\d{3})_\d+\.', re.I)
    pat_forg = re.compile(r'^\d{4}(\d{3})_\d+\.', re.I)
    rows = []
    base = root / 'OfflineSignatures'
    if not base.exists():
        deeper = [d/'OfflineSignatures' for d in root.iterdir() if d.is_dir() and (d/'OfflineSignatures').exists()]
        base   = deeper[0] if deeper else None
    if base is None:
        print('ERROR: cannot find sigComp2011 OfflineSignatures'); return pd.DataFrame(columns=['path','writer_uid','label','dataset'])
    for lang, prefix in [('Dutch','sc11d'),('Chinese','sc11c')]:
        ts = base/lang/'TrainingSet'
        if not ts.exists(): continue
        for fp in (ts/'Offline Genuine').iterdir():
            m = pat_gen.match(fp.name)
            if m: rows.append({'path':str(fp),'writer_uid':f'{prefix}_{m.group(1)}','label':'genuine','dataset':'sc11'})
        forg_dir = ts/'Offline Forgeries'
        if forg_dir.exists():
            for fp in forg_dir.iterdir():
                m = pat_forg.match(fp.name)
                if m: rows.append({'path':str(fp),'writer_uid':f'{prefix}_{m.group(1)}','label':'forgery','dataset':'sc11'})
    return pd.DataFrame(rows, columns=['path','writer_uid','label','dataset'])

def scan_sigcomp2009(root):
    root = Path(root)
    pat  = re.compile(r'^NISDCC-(\d+)_', re.I)
    rows = []
    inner = root/'NISDCC-offline-all-001-051-6g'/'NISDCC-offline-all-001-051-6g'
    target = inner if inner.exists() else root
    for fp in target.iterdir():
        if not fp.is_file(): continue
        m = pat.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'sc09_{int(m.group(1)):03d}','label':'genuine','dataset':'sc09'})
    return pd.DataFrame(rows, columns=['path','writer_uid','label','dataset'])

def scan_private(root):
    root     = Path(root)
    pat_gen  = re.compile(r'^original_(\d+)_\d+\.jpg$',   re.I)
    pat_forg = re.compile(r'^forgeries_(\d+)_\d+\.jpg$', re.I)
    rows = []
    for fp in (root/'full_org').iterdir():
        m = pat_gen.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'prv_{m.group(1)}','label':'genuine','dataset':'private'})
    for fp in (root/'full_forg').iterdir():
        m = pat_forg.match(fp.name)
        if m: rows.append({'path':str(fp),'writer_uid':f'prv_{m.group(1)}','label':'forgery','dataset':'private'})
    return pd.DataFrame(rows, columns=['path','writer_uid','label','dataset'])

print('Scanning datasets...')
df_cedar = scan_cedar(DS_ROOT/'CEDAR')
df_gpds  = scan_gpds150(DS_ROOT/'GPDS150')
df_sc11  = scan_sigcomp2011(DS_ROOT/'sigComp2011-trainingSet')
df_sc09  = scan_sigcomp2009(DS_ROOT/'SigComp2009-training')
df_prv   = scan_private(DS_ROOT/'private_signature_verification')

for name, df in [('CEDAR',df_cedar),('GPDS150',df_gpds),('SigComp2011',df_sc11),('SigComp2009',df_sc09),('Private',df_prv)]:
    gen  = (df['label']=='genuine').sum(); forg = (df['label']=='forgery').sum()
    print(f'{name:12s}  writers={df["writer_uid"].nunique():4d}  genuine={gen:6d}  forgery={forg:6d}')


---
## Section 3 -- Writer Split (70 / 10 / 20) -- same seed as v7

In [ ]:
def split_writers(writer_uids, seed=42, train_frac=0.70, val_frac=0.10):
    arr = np.array(sorted(writer_uids))
    rng = np.random.default_rng(seed)
    rng.shuffle(arr)
    n    = len(arr)
    n_tr = int(round(n * train_frac))
    n_v  = int(round(n * val_frac))
    return set(arr[:n_tr]), set(arr[n_tr:n_tr+n_v]), set(arr[n_tr+n_v:])

cedar_train, cedar_val, cedar_test = split_writers(df_cedar['writer_uid'].unique())
gpds_train,  gpds_val,  gpds_test  = split_writers(df_gpds['writer_uid'].unique())
sc11_train,  sc11_val,  sc11_test  = split_writers(df_sc11['writer_uid'].unique())
prv_train,   prv_val,   prv_test   = split_writers(df_prv['writer_uid'].unique())
sc09_train   = set(df_sc09['writer_uid'].unique())

all_train_wids  = cedar_train | gpds_train | sc11_train | sc09_train | prv_train
sorted_train    = sorted(all_train_wids)
writer_to_class = {wid: i for i, wid in enumerate(sorted_train)}
NUM_CLASSES     = len(sorted_train)

print(f'  CEDAR      train={len(cedar_train):4d}  val={len(cedar_val):3d}  test={len(cedar_test):3d}')
print(f'  GPDS150    train={len(gpds_train):4d}  val={len(gpds_val):3d}  test={len(gpds_test):3d}')
print(f'  SigComp11  train={len(sc11_train):4d}  val={len(sc11_val):3d}  test={len(sc11_test):3d}')
print(f'  SigComp09  train={len(sc09_train):4d}  (no test)')
print(f'  Private    train={len(prv_train):4d}  val={len(prv_val):3d}  test={len(prv_test):3d}')
print(f'TOTAL training classes: {NUM_CLASSES}')

---
## Section 4 -- Transforms, Datasets, Balanced Sampler

### Key change: Dataset-balanced batch sampler
Within each batch, each of the 5 datasets contributes **equally** regardless of raw image count.
Within each dataset's quota, per-sample EMA hard mining still applies.

```
v7 batch composition:          v8 batch composition:
  Private     87%                Cedar     20%
  GPDS150      5%                GPDS150   20%
  SigComp09    5%                SigComp09 20%
  Cedar        2%                SigComp11 20%
  SigComp11    1%                Private   20%
```


In [ ]:
import torchvision
_tv = tuple(int(x) for x in torchvision.__version__.split('.')[:2])

train_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=8, translate=(0.05,0.05), scale=(0.93,1.07), fill=255),
    T.RandomPerspective(distortion_scale=0.2, p=0.4, fill=255),
    *([T.ElasticTransform(alpha=30.0, sigma=4.0, fill=255)] if _tv >= (0,14) else []),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]),
    T.Lambda(lambda x: x + 0.015*torch.randn_like(x)),
])
eval_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]),
])
tta_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=5, translate=(0.03,0.03), scale=(0.96,1.04), fill=255),
    T.RandomPerspective(distortion_scale=0.1, p=0.5, fill=255),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]),
])

class SignatureClassDataset(Dataset):
    """Returns (img, class_idx, sample_idx). Stores 'dataset' column for balanced sampler."""
    def __init__(self, df, writer_to_class, writer_uid_set, transform=None):
        df_all = pd.concat([df_cedar, df_gpds, df_sc11, df_sc09, df_prv], ignore_index=True)
        sub = df_all[(df_all['writer_uid'].isin(writer_uid_set)) & (df_all['label']=='genuine')]
        self.records   = sub[['path','writer_uid','dataset']].reset_index(drop=True)
        self.w2c       = writer_to_class
        self.transform = transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img = Image.open(row['path']).convert('L')
        if self.transform: img = self.transform(img)
        return img, self.w2c[row['writer_uid']], idx

def make_balanced_mining_loader(ema_weights):
    """Dataset-balanced sampler: each dataset gets equal total weight; EMA mining within."""
    datasets  = train_ds.records['dataset'].values
    unique_ds = np.unique(datasets)
    ema_np    = ema_weights.numpy()
    combined  = np.zeros(len(train_ds))
    for ds in unique_ds:
        mask   = datasets == ds
        ema_ds = ema_np[mask]
        # Normalize EMA within this dataset, then scale by 1/n_datasets
        combined[mask] = (ema_ds / ema_ds.sum()) / len(unique_ds)
    weights = torch.tensor(combined / combined.sum(), dtype=torch.float)
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    return DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0, pin_memory=True)

train_ds        = SignatureClassDataset(None, writer_to_class, all_train_wids, transform=train_tfm)
n_samples       = len(train_ds)
sample_loss_ema = torch.ones(n_samples)

# Show dataset balance in training set
ds_counts = train_ds.records['dataset'].value_counts()
print('Training sample counts per dataset:')
for ds, cnt in ds_counts.items():
    print(f'  {ds:10s}: {cnt:6d}  ({cnt/n_samples:.1%})')
print(f'TOTAL: {n_samples:,}')

---
## Section 5 -- Phase 3: Batch-Hard Triplet Mining Setup

Each Phase-3 batch contains `N_WRITERS_P3` writers × `(N_GEN_P3 genuine + N_FORG_P3 forgery)`.
The loss mines the **hardest positive** (most distant genuine pair) and
**hardest negative** (closest genuine-forgery pair) for each genuine anchor within the batch.

```
Batch (10 writers × 5 images = 50 total):
  Writer A: gen1  gen2  gen3 | forg1  forg2
  Writer B: gen1  gen2  gen3 | forg1  forg2
  ...
  Writer J: gen1  gen2  gen3 | forg1  forg2

For anchor = A.gen1:
  hardest positive = max_dist(A.gen1, {A.gen2, A.gen3})
  hardest negative = min_dist(A.gen1, {A.forg1, A.forg2})
  loss = relu(hard_pos_dist - hard_neg_dist + margin)
```


In [ ]:
class TripletMiningDataset(Dataset):
    """Dataset for Phase 3. Returns (img, writer_uid_str, is_genuine_int)."""
    def __init__(self, df, writer_uid_set, transform=None):
        sub  = df[df['writer_uid'].isin(writer_uid_set)]
        gen  = sub[sub['label']=='genuine'].groupby('writer_uid')['path'].apply(list)
        forg = sub[sub['label']=='forgery'].groupby('writer_uid')['path'].apply(list)
        wids = [w for w in gen.index if w in forg.index and len(forg[w])>0 and len(gen[w])>=2]
        self.items     = []
        self.gen_idx   = {}
        self.forg_idx  = {}
        for w in wids:
            gi0 = len(self.items)
            for p in gen[w]:  self.items.append((p, w, 1));
            gi1 = len(self.items)
            for p in forg[w]: self.items.append((p, w, 0))
            fi1 = len(self.items)
            self.gen_idx[w]  = list(range(gi0, gi1))
            self.forg_idx[w] = list(range(gi1, fi1))
        self.eligible  = list(wids)
        self.transform = transform
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        path, wid, lbl = self.items[idx]
        img = Image.open(path).convert('L')
        if self.transform: img = self.transform(img)
        return img, wid, lbl

class StructuredBatchSampler(Sampler):
    """Yields batches where each batch has N_W writers × (N_G genuine + N_F forgery)."""
    def __init__(self, dataset, n_writers, n_gen, n_forg, seed=42):
        self.n_w      = n_writers
        self.n_g      = n_gen
        self.n_f      = n_forg
        self.seed     = seed
        self.eligible = dataset.eligible
        self.gen_idx  = {w: np.array(dataset.gen_idx[w])  for w in self.eligible}
        self.forg_idx = {w: np.array(dataset.forg_idx[w]) for w in self.eligible}
    def __len__(self):
        return len(self.eligible) // self.n_w
    def __iter__(self):
        rng  = np.random.default_rng(self.seed)
        shuf = rng.permutation(self.eligible)
        for i in range(0, len(shuf) - self.n_w + 1, self.n_w):
            batch_idx = []
            for w in shuf[i:i+self.n_w]:
                gi = rng.choice(self.gen_idx[w],  size=min(self.n_g, len(self.gen_idx[w])),  replace=False)
                fi = rng.choice(self.forg_idx[w], size=min(self.n_f, len(self.forg_idx[w])), replace=True)
                batch_idx.extend(gi.tolist())
                batch_idx.extend(fi.tolist())
            yield batch_idx

def collate_triplet(batch):
    imgs = torch.stack([b[0] for b in batch])
    wids = [b[1] for b in batch]
    lbls = [int(b[2]) for b in batch]
    return imgs, wids, lbls

def batch_hard_triplet_loss(emb, wids, is_gen_list, margin=0.3):
    """Batch-hard triplet: anchor=genuine, pos=hardest genuine, neg=hardest forgery."""
    n     = len(emb)
    emb_n = F.normalize(emb, dim=1)
    cdist = 1 - (emb_n @ emb_n.t())   # (B,B) cosine distances; clamp to avoid fp noise
    cdist = cdist.clamp(min=0.)
    is_gen = [bool(x) for x in is_gen_list]
    # Build writer-same matrix (string comparison, cheap for n=50)
    same_w = torch.zeros(n, n, dtype=torch.bool, device=emb.device)
    for i in range(n):
        for j in range(n):
            same_w[i, j] = (wids[i] == wids[j])
    is_gen_t = torch.tensor(is_gen, dtype=torch.bool, device=emb.device)
    total = torch.zeros(1, device=emb.device)
    count = 0
    for i in range(n):
        if not is_gen[i]: continue
        pos_mask = same_w[i] & is_gen_t; pos_mask[i] = False
        neg_mask = same_w[i] & (~is_gen_t)
        if not pos_mask.any() or not neg_mask.any(): continue
        hp = cdist[i][pos_mask].max()
        hn = cdist[i][neg_mask].min()
        total = total + F.relu(hp - hn + margin)
        count += 1
    return total.squeeze() / max(count, 1)

# Build Phase 3 triplet dataset (GPDS + CEDAR + Private training writers with forgeries)
df_for_p3  = pd.concat([
    df_gpds [df_gpds ['writer_uid'].isin(gpds_train)],
    df_cedar[df_cedar['writer_uid'].isin(cedar_train)],
    df_prv  [df_prv  ['writer_uid'].isin(prv_train)],
], ignore_index=True)
p3_wids    = gpds_train | cedar_train | prv_train
triplet_ds = TripletMiningDataset(df_for_p3, p3_wids, transform=train_tfm)
print(f'Phase 3 triplet dataset: {len(triplet_ds):,} items from {len(triplet_ds.eligible)} eligible writers')
print(f'Batch structure: {N_WRITERS_P3} writers × ({N_GEN_P3} gen + {N_FORG_P3} forg) = {BATCH_P3} images/batch')
print(f'Batches per epoch: ~{len(triplet_ds.eligible)//N_WRITERS_P3}')

---
## Section 6 -- Model Architecture (unchanged from v7)

In [ ]:
class SubCenterArcFace(nn.Module):
    def __init__(self, emb_dim, n_classes, K=3, scale=64.0, margin=0.5):
        super().__init__()
        self.K = K; self.scale = scale
        self.weight = nn.Parameter(torch.FloatTensor(n_classes * K, emb_dim))
        nn.init.xavier_uniform_(self.weight)
        self._update_margin(margin)
    def _update_margin(self, m):
        self.margin = m
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m); self.mm = math.sin(math.pi - m) * m
    def forward(self, emb, labels, reduction='mean'):
        emb_n = F.normalize(emb, dim=1); W_n = F.normalize(self.weight, dim=1)
        cos_all = F.linear(emb_n, W_n)                              # (B, n_classes*K)
        cos_all = cos_all.view(-1, cos_all.shape[1]//self.K, self.K)
        cos_t, _ = cos_all.max(dim=2)                               # (B, n_classes)
        sin_t  = torch.sqrt((1 - cos_t**2).clamp(min=1e-6))
        cos_tm = cos_t * self.cos_m - sin_t * self.sin_m
        cos_tm = torch.where(cos_t > self.th, cos_tm, cos_t - self.mm)
        one_hot = torch.zeros_like(cos_t)
        one_hot.scatter_(1, labels.view(-1,1).long(), 1)
        logits = (one_hot * cos_tm + (1-one_hot) * cos_t) * self.scale
        return F.cross_entropy(logits, labels.long(), reduction=reduction)

class GeMPool(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1)*p); self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1./self.p)

class MultiScaleResNet34(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        try:
            from torchvision.models import ResNet34_Weights
            net = models.resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        except (ImportError, AttributeError):
            net = models.resnet34(pretrained=True)
        w = net.conv1.weight.data
        net.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.conv1.weight.data = w.mean(dim=1, keepdim=True)
        self.stem   = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1 = net.layer1; self.layer2 = net.layer2
        self.layer3 = net.layer3; self.layer4 = net.layer4
        self.gem3   = GeMPool(p=3); self.gem4 = GeMPool(p=3)
        self.head   = nn.Sequential(
            nn.BatchNorm1d(768), nn.Dropout(0.4),
            nn.Linear(768, emb_dim), nn.BatchNorm1d(emb_dim, affine=False),
        )
    def forward(self, x):
        x  = self.stem(x); x = self.layer1(x); x = self.layer2(x)
        x3 = self.layer3(x); x4 = self.layer4(x3)
        return self.head(torch.cat([self.gem3(x3).flatten(1), self.gem4(x4).flatten(1)], dim=1))

set_seed(SEED)
backbone = MultiScaleResNet34(EMB_DIM).to(DEVICE)
arc_head = SubCenterArcFace(EMB_DIM, NUM_CLASSES, K=K_SUB, scale=64.0, margin=0.1).to(DEVICE)
n_bb = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
n_hd = sum(p.numel() for p in arc_head.parameters()  if p.requires_grad)
print(f'Backbone params: {n_bb:,}  |  ArcFace params: {n_hd:,}  ({NUM_CLASSES}×{K_SUB}×{EMB_DIM})')
x_s, y_s, _ = next(iter(DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)))
with torch.no_grad():
    emb_s = backbone(x_s.to(DEVICE))
print(f'Embedding shape: {emb_s.shape}  |  gem3.p={backbone.gem3.p.item():.3f}')

---
## Section 7 -- Phase 1: Full Network Training

**New**: dataset-balanced sampler + margin 0.10→**0.70** over **20** epochs + **40** total epochs.


In [ ]:
set_seed(SEED)
backbone = MultiScaleResNet34(EMB_DIM).to(DEVICE)
arc_head = SubCenterArcFace(EMB_DIM, NUM_CLASSES, K=K_SUB, scale=64.0, margin=0.1).to(DEVICE)

params_p1    = list(backbone.parameters()) + list(arc_head.parameters())
optimizer_p1 = torch.optim.Adam(params_p1, lr=1e-3, weight_decay=1e-4)
sched_p1     = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p1, T_max=N_P1-WARMUP, eta_min=1e-6)
sample_loss_ema = torch.ones(n_samples)
hist_p1 = {'loss':[], 'acc':[]}

for epoch in range(N_P1):
    if epoch < WARMUP:
        for g in optimizer_p1.param_groups: g['lr'] = 1e-3*(epoch+1)/WARMUP
    m = MARGIN_START + (MARGIN_END-MARGIN_START)*min(epoch,RAMP_EPOCHS)/RAMP_EPOCHS
    arc_head._update_margin(m)

    loader = make_balanced_mining_loader(sample_loss_ema)
    backbone.train(); arc_head.train()
    ep_loss = 0.; correct = 0; total = 0; n_b = 0

    for imgs, labels, idxs in tqdm(loader, desc=f'P1 {epoch+1:02d}/{N_P1}', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        emb      = backbone(imgs)
        per_loss = arc_head(emb, labels, reduction='none')
        loss     = per_loss.mean()
        optimizer_p1.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params_p1, 1.0)
        optimizer_p1.step()
        with torch.no_grad():
            idx_cpu = idxs.cpu(); lv = per_loss.detach().cpu()
            sample_loss_ema[idx_cpu] = 0.7*sample_loss_ema[idx_cpu] + 0.3*lv
            cos_all = F.linear(F.normalize(emb,dim=1), F.normalize(arc_head.weight,dim=1))
            logits  = cos_all.view(-1,NUM_CLASSES,K_SUB).max(dim=2).values*arc_head.scale
            correct += (logits.argmax(1)==labels.long()).sum().item(); total += labels.size(0)
        ep_loss += loss.item(); n_b += 1

    if epoch >= WARMUP: sched_p1.step()
    avg = ep_loss/n_b; acc = correct/total
    hist_p1['loss'].append(avg); hist_p1['acc'].append(acc)
    print(f'P1 E{epoch+1:02d}/{N_P1}  loss={avg:.4f}  acc={acc:.3f}  '
          f'lr={optimizer_p1.param_groups[0]["lr"]:.2e}  margin={m:.3f}')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,4))
e1 = range(1,len(hist_p1['loss'])+1)
axes[0].plot(e1,hist_p1['loss']); axes[0].set_title('Phase 1 Loss'); axes[0].grid(True)
axes[1].plot(e1,hist_p1['acc']);  axes[1].set_title('Phase 1 Accuracy'); axes[1].grid(True)
plt.tight_layout(); plt.show()

---
## Section 8 -- Phase 2: Fine-Tuning (20 epochs, frozen stem+layer1-3)

In [ ]:
for name, p in backbone.named_parameters():
    p.requires_grad = not any(name.startswith(s) for s in ['stem.','layer1','layer2','layer3'])
print(f'Frozen: {sum(1 for p in backbone.parameters() if not p.requires_grad)}  '
      f'| Trainable: {sum(1 for p in backbone.parameters() if p.requires_grad)}')

optimizer_p2 = torch.optim.Adam([
    {'params': [p for p in backbone.parameters() if p.requires_grad], 'lr': 2e-4},
    {'params': arc_head.parameters(), 'lr': 5e-4},
], weight_decay=1e-4)
sched_p2  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=N_P2, eta_min=1e-6)
params_p2 = [p for p in backbone.parameters() if p.requires_grad] + list(arc_head.parameters())
hist_p2   = {'loss':[], 'acc':[]}

for epoch in range(N_P2):
    loader = make_balanced_mining_loader(sample_loss_ema)
    backbone.train(); arc_head.train()
    ep_loss = 0.; correct = 0; total = 0; n_b = 0
    for imgs, labels, idxs in tqdm(loader, desc=f'P2 {epoch+1:02d}/{N_P2}', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        emb      = backbone(imgs)
        per_loss = arc_head(emb, labels, reduction='none')
        loss     = per_loss.mean()
        optimizer_p2.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params_p2, 1.0); optimizer_p2.step()
        with torch.no_grad():
            idx_cpu = idxs.cpu(); lv = per_loss.detach().cpu()
            sample_loss_ema[idx_cpu] = 0.7*sample_loss_ema[idx_cpu] + 0.3*lv
            cos_all = F.linear(F.normalize(emb,dim=1), F.normalize(arc_head.weight,dim=1))
            logits  = cos_all.view(-1,NUM_CLASSES,K_SUB).max(dim=2).values*arc_head.scale
            correct += (logits.argmax(1)==labels.long()).sum().item(); total += labels.size(0)
        ep_loss += loss.item(); n_b += 1
    sched_p2.step()
    avg = ep_loss/n_b; acc = correct/total
    hist_p2['loss'].append(avg); hist_p2['acc'].append(acc)
    print(f'P2 E{epoch+1:02d}/{N_P2}  loss={avg:.4f}  acc={acc:.3f}  lr={sched_p2.get_last_lr()[0]:.2e}')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,4))
e2 = range(1,len(hist_p2['loss'])+1)
axes[0].plot(e2,hist_p2['loss']); axes[0].set_title('Phase 2 Loss'); axes[0].grid(True)
axes[1].plot(e2,hist_p2['acc']);  axes[1].set_title('Phase 2 Accuracy'); axes[1].grid(True)
plt.tight_layout(); plt.show()

---
## Section 9 -- Phase 3: Batch-Hard Triplet Mining (15 epochs)

Each batch is structured so every genuine anchor is guaranteed to have at least 2 genuine
positives and 2 forgery negatives from the **same writer** in the batch.
Loss mines the **hardest** (most surprising) case per anchor.


In [ ]:
for p in backbone.parameters(): p.requires_grad = True
params_p3    = list(backbone.parameters())
optimizer_p3 = torch.optim.Adam(params_p3, lr=5e-5, weight_decay=1e-4)
sched_p3     = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p3, T_max=N_P3, eta_min=1e-7)
hist_p3      = {'loss':[], 'n_tri':[]}

for epoch in range(N_P3):
    # Fresh shuffle each epoch by re-initialising the sampler with a new seed
    sampler_p3 = StructuredBatchSampler(triplet_ds, N_WRITERS_P3, N_GEN_P3, N_FORG_P3, seed=SEED+epoch)
    ld_p3      = DataLoader(triplet_ds, batch_sampler=sampler_p3,
                            collate_fn=collate_triplet, num_workers=0)
    backbone.train()
    ep_loss = 0.; ep_tri = 0; n_b = 0

    for imgs, wids_b, lbls_b in tqdm(ld_p3, desc=f'P3 {epoch+1:02d}/{N_P3}', leave=False):
        imgs = imgs.to(DEVICE)
        emb  = backbone(imgs)
        loss = batch_hard_triplet_loss(emb, wids_b, lbls_b, margin=0.3)
        optimizer_p3.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params_p3, 1.0); optimizer_p3.step()
        ep_loss += loss.item(); n_b += 1

    sched_p3.step()
    avg = ep_loss/n_b if n_b else float('nan')
    hist_p3['loss'].append(avg)
    print(f'P3 E{epoch+1:02d}/{N_P3}  triplet_loss={avg:.4f}  lr={sched_p3.get_last_lr()[0]:.2e}')

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(range(1,len(hist_p3['loss'])+1), hist_p3['loss'])
ax.set_title('Phase 3 Batch-Hard Triplet Loss'); ax.set_xlabel('Epoch'); ax.grid(True)
plt.tight_layout(); plt.show()

---
## Section 10 -- Save Model Weights

In [ ]:
import os
models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(exist_ok=True)

backbone_path = models_dir / 'v8_backbone.pth'
archead_path  = models_dir / 'v8_archead.pth'
torch.save(backbone.state_dict(), backbone_path)
torch.save(arc_head.state_dict(), archead_path)

print(f'Backbone saved : {backbone_path}  ({backbone_path.stat().st_size/1e6:.1f} MB)')
print(f'ArcFace saved  : {archead_path}  ({archead_path.stat().st_size/1e6:.1f} MB)')
print()
print('To reload later:')
print('  backbone = MultiScaleResNet34(EMB_DIM).to(DEVICE)')
print(f'  backbone.load_state_dict(torch.load("{backbone_path}", map_location=DEVICE))')

---
## Section 11 -- Evaluation Infrastructure

### Two evaluation protocols run side-by-side:

**Pairwise + TTA**: same as v7 but each image embedded N=8 times; embeddings averaged.
**Prototype + TTA**: K_ENROLL=6 genuine images → L2-normalised mean embedding (writer prototype).
Remaining genuine + all forgeries compared against prototype.

All unique test images are embedded **once** into a cache, then both protocols use the cache.


In [ ]:
def build_embedding_cache(df, wid_set, backbone, device, tfm, n_views=TTA_N, imgs_per_batch=8):
    """Pre-embed all unique images for test writers with TTA. Returns {path: tensor(256,)}."""
    backbone.eval()
    sub   = df[df['writer_uid'].isin(wid_set)]
    paths = sub['path'].unique().tolist()
    cache = {}
    with torch.no_grad():
        for i in tqdm(range(0, len(paths), imgs_per_batch), desc='TTA embed', leave=False):
            batch_paths = paths[i:i+imgs_per_batch]
            views = []
            for p in batch_paths:
                img = Image.open(p).convert('L')
                views.extend([tfm(img) for _ in range(n_views)])
            views = torch.stack(views).to(device)                    # (B*n_views, C, H, W)
            embs  = F.normalize(backbone(views), dim=1)              # (B*n_views, 256)
            embs  = embs.view(len(batch_paths), n_views, -1).mean(1) # (B, 256)
            embs  = F.normalize(embs, dim=1).cpu()
            for j, p in enumerate(batch_paths):
                cache[p] = embs[j]
    return cache

def build_pools(df, wid_set):
    gen, forg = {}, {}
    for wid, grp in df[df['writer_uid'].isin(wid_set)].groupby('writer_uid'):
        g = grp[grp['label']=='genuine']['path'].tolist()
        f = grp[grp['label']=='forgery']['path'].tolist()
        if g: gen[wid]  = g
        if f: forg[wid] = f
    return gen, forg

def generate_pairs(df, wid_set, n_pairs=10000, seed=1, neg_mix=0.8):
    gen, forg = build_pools(df, wid_set)
    writers = sorted(gen); wf = sorted(set(gen)&set(forg))
    rng = np.random.default_rng(seed)
    n_pos=n_pairs//2; n_neg=n_pairs-n_pos
    n_sf=int(round(n_neg*neg_mix)); n_rnd=n_neg-n_sf
    rows=[]
    for _ in range(n_pos):
        w=rng.choice(writers); g=gen[w]
        if len(g)<2: continue
        i,j=rng.choice(len(g),2,replace=False); rows.append({'path_a':g[i],'path_b':g[j],'label':1})
    for _ in range(n_sf):
        if not wf: break
        w=rng.choice(wf); g=gen[w]; f=forg[w]
        rows.append({'path_a':g[rng.integers(len(g))],'path_b':f[rng.integers(len(f))],'label':0})
    for _ in range(n_rnd):
        if len(writers)<2: break
        w1,w2=rng.choice(writers,2,replace=False)
        rows.append({'path_a':gen[w1][rng.integers(len(gen[w1]))],'path_b':gen[w2][rng.integers(len(gen[w2]))],'label':0})
    return pd.DataFrame(rows)

_NAN_RES = {'eer':float('nan'),'auc':float('nan'),'accuracy':float('nan')}

def run_eval_pairwise(name, df, test_wids, cache, n_pairs=10000):
    """Pairwise evaluation using TTA embedding cache."""
    if not test_wids:
        print(f'=== {name}: SKIPPED (no test writers)'); return _NAN_RES, float('nan'), np.array([]), np.array([])
    pairs_df = generate_pairs(df, test_wids, n_pairs=n_pairs, seed=1)
    valid    = pairs_df[pairs_df['path_a'].isin(cache) & pairs_df['path_b'].isin(cache)]
    if len(valid) < 10:
        print(f'=== {name}: too few valid pairs ({len(valid)})'); return _NAN_RES, float('nan'), np.array([]), np.array([])
    embs_a = torch.stack([cache[p] for p in valid['path_a']])
    embs_b = torch.stack([cache[p] for p in valid['path_b']])
    sims   = (embs_a * embs_b).sum(dim=1).numpy()
    ys     = valid['label'].values
    res    = compute_metrics(ys, sims)
    gap    = sims[ys==1].mean() - sims[ys==0].mean()
    print(f'  [pairwise+TTA]  EER={res["eer"]:.2%}  AUC={res["auc"]:.4f}  Acc={res["accuracy"]:.2%}  gap={gap:.4f}')
    return res, gap, sims, ys

def run_eval_prototype(name, df, test_wids, cache, n_enroll=K_ENROLL):
    """Prototype evaluation: writer prototype = mean of K_ENROLL enrolled genuine embeds."""
    if not test_wids:
        print(f'=== {name}: SKIPPED (no test writers)'); return _NAN_RES, float('nan'), np.array([]), np.array([])
    scores, ys = [], []
    n_valid = 0
    for wid in sorted(test_wids):
        sub        = df[df['writer_uid']==wid]
        gen_paths  = [p for p in sub[sub['label']=='genuine']['path'].tolist() if p in cache]
        forg_paths = [p for p in sub[sub['label']=='forgery']['path'].tolist() if p in cache]
        if len(gen_paths) < n_enroll+1: continue
        # Prototype from first n_enroll genuine
        proto = F.normalize(torch.stack([cache[p] for p in gen_paths[:n_enroll]]).mean(0), dim=0)
        # Genuine queries (positive)
        for p in gen_paths[n_enroll:]:
            scores.append((proto * cache[p]).sum().item()); ys.append(1)
        # Forgery queries (negative)
        for p in forg_paths:
            scores.append((proto * cache[p]).sum().item()); ys.append(0)
        n_valid += 1
    scores, ys = np.array(scores), np.array(ys)
    if len(scores) < 10:
        return _NAN_RES, float('nan'), scores, ys
    res = compute_metrics(ys, scores)
    gap = scores[ys==1].mean() - scores[ys==0].mean()
    print(f'  [prototype+TTA] EER={res["eer"]:.2%}  AUC={res["auc"]:.4f}  Acc={res["accuracy"]:.2%}  gap={gap:.4f}  (enroll={n_enroll}, writers={n_valid})')
    return res, gap, scores, ys

---
## Section 12 -- CEDAR Test Evaluation

In [ ]:
print('Building CEDAR embedding cache (TTA=8)...')
cache_cedar = build_embedding_cache(df_cedar, cedar_test, backbone, DEVICE, tta_tfm, TTA_N)
print(f'=== CEDAR (n_test_writers={len(cedar_test)}) ===')
res_cedar_pw,  gap_cedar_pw,  sims_cedar_pw,  y_cedar_pw  = run_eval_pairwise ('CEDAR', df_cedar, cedar_test, cache_cedar)
res_cedar_pro, gap_cedar_pro, sims_cedar_pro, y_cedar_pro = run_eval_prototype('CEDAR', df_cedar, cedar_test, cache_cedar)

fig, ax = plt.subplots(figsize=(6,5))
if len(y_cedar_pw)>0:
    fpr,tpr,_ = sk_metrics.roc_curve(y_cedar_pw, sims_cedar_pw)
    ax.plot(fpr,tpr,label=f'Pairwise+TTA AUC={res_cedar_pw["auc"]:.4f}')
if len(y_cedar_pro)>0:
    fpr,tpr,_ = sk_metrics.roc_curve(y_cedar_pro, sims_cedar_pro)
    ax.plot(fpr,tpr,label=f'Prototype+TTA AUC={res_cedar_pro["auc"]:.4f}')
ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FAR'); ax.set_ylabel('TAR')
ax.set_title('ROC -- CEDAR'); ax.legend(); ax.grid(True); plt.show()

---
## Section 13 -- GPDS150 Test Evaluation

In [ ]:
print('Building GPDS150 embedding cache (TTA=8)...')
cache_gpds = build_embedding_cache(df_gpds, gpds_test, backbone, DEVICE, tta_tfm, TTA_N)
print(f'=== GPDS150 (n_test_writers={len(gpds_test)}) ===')
res_gpds_pw,  gap_gpds_pw,  sims_gpds_pw,  y_gpds_pw  = run_eval_pairwise ('GPDS150', df_gpds, gpds_test, cache_gpds)
res_gpds_pro, gap_gpds_pro, sims_gpds_pro, y_gpds_pro = run_eval_prototype('GPDS150', df_gpds, gpds_test, cache_gpds)

fig, ax = plt.subplots(figsize=(6,5))
if len(y_gpds_pw)>0:
    fpr,tpr,_ = sk_metrics.roc_curve(y_gpds_pw, sims_gpds_pw)
    ax.plot(fpr,tpr,label=f'Pairwise+TTA AUC={res_gpds_pw["auc"]:.4f}')
if len(y_gpds_pro)>0:
    fpr,tpr,_ = sk_metrics.roc_curve(y_gpds_pro, sims_gpds_pro)
    ax.plot(fpr,tpr,label=f'Prototype+TTA AUC={res_gpds_pro["auc"]:.4f}')
ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FAR'); ax.set_ylabel('TAR')
ax.set_title('ROC -- GPDS150'); ax.legend(); ax.grid(True); plt.show()

---
## Section 14 -- SigComp2011 Test Evaluation

In [ ]:
print('SigComp2011 test writers:')
for wid in sorted(sc11_test):
    g = (df_sc11[(df_sc11['writer_uid']==wid)&(df_sc11['label']=='genuine')]).shape[0]
    f = (df_sc11[(df_sc11['writer_uid']==wid)&(df_sc11['label']=='forgery')]).shape[0]
    print(f'  {wid}  genuine={g}  forgery={f}')
cache_sc11 = build_embedding_cache(df_sc11, sc11_test, backbone, DEVICE, tta_tfm, TTA_N)
print(f'=== SigComp2011 (n_test_writers={len(sc11_test)}) ===')
res_sc11_pw,  gap_sc11_pw,  sims_sc11_pw,  y_sc11_pw  = run_eval_pairwise ('SigComp2011', df_sc11, sc11_test, cache_sc11, n_pairs=3000)
res_sc11_pro, gap_sc11_pro, sims_sc11_pro, y_sc11_pro = run_eval_prototype('SigComp2011', df_sc11, sc11_test, cache_sc11)

fig, ax = plt.subplots(figsize=(6,5))
if len(y_sc11_pw)>0:
    fpr,tpr,_ = sk_metrics.roc_curve(y_sc11_pw, sims_sc11_pw)
    ax.plot(fpr,tpr,label=f'Pairwise+TTA AUC={res_sc11_pw["auc"]:.4f}')
if len(y_sc11_pro)>0:
    fpr,tpr,_ = sk_metrics.roc_curve(y_sc11_pro, sims_sc11_pro)
    ax.plot(fpr,tpr,label=f'Prototype+TTA AUC={res_sc11_pro["auc"]:.4f}')
ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FAR'); ax.set_ylabel('TAR')
ax.set_title('ROC -- SigComp2011'); ax.legend(); ax.grid(True); plt.show()

---
## Section 15 -- Private Dataset Test Evaluation (**PRIMARY THESIS METRIC**)

800 test writers × 12 genuine + 16 forgery. Enroll 6 genuine → prototype.


In [ ]:
print(f'Building Private embedding cache ({len(prv_test)} writers, TTA={TTA_N})...')
cache_prv = build_embedding_cache(df_prv, prv_test, backbone, DEVICE, tta_tfm, TTA_N)
print(f'=== Private (n_test_writers={len(prv_test)}) ===')
res_prv_pw,  gap_prv_pw,  sims_prv_pw,  y_prv_pw  = run_eval_pairwise ('Private', df_prv, prv_test, cache_prv)
res_prv_pro, gap_prv_pro, sims_prv_pro, y_prv_pro = run_eval_prototype('Private', df_prv, prv_test, cache_prv)

fig, ax = plt.subplots(figsize=(6,5))
if len(y_prv_pw)>0:
    fpr,tpr,_ = sk_metrics.roc_curve(y_prv_pw, sims_prv_pw)
    ax.plot(fpr,tpr,label=f'Pairwise+TTA AUC={res_prv_pw["auc"]:.4f}')
if len(y_prv_pro)>0:
    fpr,tpr,_ = sk_metrics.roc_curve(y_prv_pro, sims_prv_pro)
    ax.plot(fpr,tpr,label=f'Prototype+TTA AUC={res_prv_pro["auc"]:.4f}')
ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FAR'); ax.set_ylabel('TAR')
ax.set_title('ROC -- Private dataset (thesis evaluation)'); ax.legend(); ax.grid(True); plt.show()

---
## Section 16 -- Combined (Pooled) Evaluation -- Prototype+TTA

In [ ]:
# Pool prototype scores across all datasets
_all_sims, _all_y = [], []
for s, y in [(sims_cedar_pro,y_cedar_pro),(sims_gpds_pro,y_gpds_pro),
             (sims_prv_pro,y_prv_pro),(sims_sc11_pro,y_sc11_pro)]:
    if len(s)>0: _all_sims.append(s); _all_y.append(y)
sims_comb = np.concatenate(_all_sims); y_comb = np.concatenate(_all_y)
res_comb  = compute_metrics(y_comb, sims_comb)
gap_comb  = sims_comb[y_comb==1].mean() - sims_comb[y_comb==0].mean()
print(f'=== Combined prototype+TTA ({len(sims_comb):,} pairs) ===')
print(f'  EER={res_comb["eer"]:.2%}  AUC={res_comb["auc"]:.4f}  Acc={res_comb["accuracy"]:.2%}  gap={gap_comb:.4f}')

fpr,tpr,_ = sk_metrics.roc_curve(y_comb, sims_comb)
plt.figure(figsize=(6,5))
plt.plot(fpr,tpr,label=f'Combined AUC={res_comb["auc"]:.4f}')
plt.plot([0,1],[0,1],'k--'); plt.xlabel('FAR'); plt.ylabel('TAR')
plt.title('ROC -- Combined (all datasets, prototype+TTA)'); plt.legend(); plt.grid(True); plt.show()

---
## Section 17 -- PCA: Private Test Writers (15 sampled)

In [ ]:
import random
random.seed(SEED)
pca_wids = random.sample(sorted(prv_test), min(15, len(prv_test)))
pca_w2c  = {wid:i for i,wid in enumerate(pca_wids)}
all_embs, all_cls = [], []
backbone.eval()
with torch.no_grad():
    for wid in pca_wids:
        gen_paths = df_prv[(df_prv['writer_uid']==wid)&(df_prv['label']=='genuine')]['path'].tolist()
        for p in gen_paths:
            if p in cache_prv:
                all_embs.append(cache_prv[p].numpy())
                all_cls.append(pca_w2c[wid])

all_embs = np.array(all_embs); all_cls = np.array(all_cls)
pca   = PCA(n_components=2, random_state=SEED)
e2d   = pca.fit_transform(all_embs)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}')

fig, ax = plt.subplots(figsize=(9,7))
cmap = plt.cm.get_cmap('tab20', len(pca_wids))
for i, wid in enumerate(pca_wids):
    mask = all_cls == pca_w2c[wid]
    ax.scatter(e2d[mask,0], e2d[mask,1], color=cmap(i), label=wid, s=60, alpha=0.85)
ax.set_title('PCA -- Private test writers (genuine only) -- v8')
ax.legend(loc='best', fontsize=7); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Section 18 -- Results Summary

In [ ]:
from IPython.display import Markdown, display

def f(v, fmt): return f'{v:{fmt}}' if v==v else 'N/A'
def row(name, pw, pro, gap_pw, gap_pro, n_w):
    return (f'| {name:12s} | {n_w:^13d} | {f(pw["eer"],".2%")} | {f(pw["auc"],".4f")} '
            f'| {f(pro["eer"],".2%")} | {f(pro["auc"],".4f")} | {f(gap_pro,".4f")} |')

hdr = ('| Dataset | Test writers | EER pair+TTA | AUC pair+TTA | EER proto+TTA | AUC proto+TTA | Cosine gap |\n'
       '|:--------|:------------:|:------------:|:------------:|:-------------:|:-------------:|:----------:|')
table = '\n'.join([
    row('CEDAR',       res_cedar_pw, res_cedar_pro, gap_cedar_pw, gap_cedar_pro, len(cedar_test)),
    row('GPDS150',     res_gpds_pw,  res_gpds_pro,  gap_gpds_pw,  gap_gpds_pro,  len(gpds_test)),
    row('SigComp2011', res_sc11_pw,  res_sc11_pro,  gap_sc11_pw,  gap_sc11_pro,  len(sc11_test)),
    row('Private',     res_prv_pw,   res_prv_pro,   gap_prv_pw,   gap_prv_pro,   len(prv_test)),
    row('Combined',    res_comb,     res_comb,      gap_comb,     gap_comb,
        len(cedar_test)+len(gpds_test)+len(sc11_test)+len(prv_test)),
])

display(Markdown(f'''
### v8 Results -- Balanced Sampler + Prototype+TTA

{hdr}
{table}

### v7 → v8 comparison (Private dataset, pairwise EER)
| Version | Private EER | CEDAR EER | Key change |
|:--------|:-----------:|:---------:|:-----------|
| v7 pairwise | 18.50% | 11.96% | baseline (imbalanced sampler) |
| v8 pairwise+TTA | {res_prv_pw["eer"]:.2%} | {res_cedar_pw["eer"]:.2%} | balanced sampler + larger margin + hard triplet |
| v8 prototype+TTA | {res_prv_pro["eer"]:.2%} | {res_cedar_pro["eer"]:.2%} | + prototype enrollment (6 genuine) |

### Architecture
- Backbone: MultiScaleResNet34 (GeM layer3+layer4 → 768-dim → 256-dim)
- Loss P1/P2: SubCenterArcFace K=3, margin 0.10→0.70 (20-ep ramp)
- Loss P3: Batch-hard triplet (margin=0.3), structured {N_WRITERS_P3}×{N_GEN_P3+N_FORG_P3} batches
- Training: {N_P1} + {N_P2} + {N_P3} epochs  |  Balanced dataset sampler (5 datasets, equal weight)
- Inference: TTA N={TTA_N}  |  Prototype enroll K={K_ENROLL} genuine images
- Training classes: {NUM_CLASSES}  (CEDAR+GPDS+SC11+SC09+Private)
'''))